In [1]:
# =====================================================================
# PART 1 OF 2 : KDE FEATURE-DISTRIBUTION PLOTS
#
# Input:
#     /content/drive/MyDrive/Objective2/
#
# Output:
#     /content/drive/MyDrive/Objective2/output_cascading/caskde_plots/
#
# This notebook performs data loading, preprocessing, and KDE
# feature-distribution plotting ONLY.
#
# It also saves the cleaned dataset so the companion notebook
# (classification.ipynb) can train the models without repeating
# the preprocessing.
#
# Figures:
#     - KDE-only feature distributions
#     - Blue, coffee, deep-red and pink colour palette
#     - Legend below the KDE graph
#     - No titles
#     - PDF at 600 dpi
# =====================================================================


In [2]:
# =====================================================================
# 1. MOUNT GOOGLE DRIVE
# =====================================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# =====================================================================
# 2. IMPORT LIBRARIES
# =====================================================================

from pathlib import Path
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

from IPython.display import display

warnings.filterwarnings("ignore")


In [4]:
# =====================================================================
# 3. GLOBAL SETTINGS
# =====================================================================

RANDOM_STATE = 42

# Maximum rows used per class for KDE estimation.
# This affects plotting speed only.
# The complete dataset is still used for model training.
MAX_KDE_SAMPLES = 5000

# Remove an existing output folder before creating new results.
RESET_OUTPUT_FOLDER = True

In [5]:
# =====================================================================
# 4. FONT AND FIGURE SETTINGS
# =====================================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "DejaVu Serif",
        "Computer Modern",
        "Times New Roman"
    ],
    "font.size": 20,
    "axes.titlesize": 22,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 14,
    "figure.titlesize": 22
})


# Aesthetically pleasing KDE colours
KDE_COLOURS = {
    "Benign": "#2F6690",    # muted blue
    "DNSCat2": "#8B5E3C",   # coffee brown
    "DNSTT": "#9B1B30",     # deep red
    "Iodine": "#D36C9D"     # soft pink
}

LABEL_ORDER = [
    "Benign",
    "DNSCat2",
    "DNSTT",
    "Iodine"
]


# Colours used for ROC curves
ROC_COLOURS = {
    "Logistic Regression": "#2F6690",
    "Decision Tree": "#8B5E3C",
    "Random Forest": "#9B1B30",
    "SVM": "#D36C9D"
}

In [6]:
# =====================================================================
# 5. INPUT AND OUTPUT DIRECTORIES
# =====================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Objective2"
)

OUTPUT_DIR = BASE_DIR / "KDE"

KDE_DIR = OUTPUT_DIR / "KDE"
STAGE1_DIR = OUTPUT_DIR / "KDE_stage1_results"
STAGE2_DIR = OUTPUT_DIR / "KDE_stage2_results"
CLEANED_DATA_DIR = OUTPUT_DIR / "cleaned_data"
MODEL_DIR = OUTPUT_DIR / "saved_models"


if RESET_OUTPUT_FOLDER and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)


for directory in [
    OUTPUT_DIR,
    KDE_DIR,
    STAGE1_DIR,
    STAGE2_DIR,
    CLEANED_DATA_DIR,
    MODEL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Input directory :", BASE_DIR)
print("Output directory:", OUTPUT_DIR)

Input directory : /content/drive/MyDrive/Objective2
Output directory: /content/drive/MyDrive/Objective2/KDE


In [7]:
# =====================================================================
# 6. LOCATE THE FOUR CSV FILES
# =====================================================================

csv_files = sorted(
    BASE_DIR.glob("*.csv")
)


if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files were found inside:\n{BASE_DIR}"
    )


print("\nCSV files found:")

for file_path in csv_files:
    print(" -", file_path.name)


def clean_filename(filename):
    """
    Convert a filename to lowercase alphanumeric text.
    """

    return re.sub(
        r"[^a-z0-9]",
        "",
        filename.lower()
    )


def find_csv(keyword):
    """
    Find a CSV file whose name contains the keyword.
    """

    cleaned_keyword = clean_filename(
        keyword
    )

    matches = []

    for file_path in csv_files:

        cleaned_file_name = clean_filename(
            file_path.stem
        )

        if cleaned_keyword in cleaned_file_name:
            matches.append(file_path)


    if len(matches) == 0:

        raise FileNotFoundError(
            f"No CSV file matching '{keyword}' was found "
            f"inside {BASE_DIR}"
        )


    if len(matches) > 1:

        print(
            f"Warning: multiple files matched '{keyword}'. "
            f"Using: {matches[0].name}"
        )


    return matches[0]


FILES = {
    "Benign": find_csv("benign"),
    "DNSCat2": find_csv("dnscat"),
    "DNSTT": find_csv("dnstt"),
    "Iodine": find_csv("iodine")
}


print("\nSelected files:")

for dataset_name, file_path in FILES.items():

    print(
        f"{dataset_name:8s}: {file_path.name}"
    )


CSV files found:
 - Benign.csv
 - Dnscat2.csv
 - Dnstt.csv
 - Iodine.csv

Selected files:
Benign  : Benign.csv
DNSCat2 : Dnscat2.csv
DNSTT   : Dnstt.csv
Iodine  : Iodine.csv


In [8]:
# =====================================================================
# 7. HELPER FUNCTIONS FOR COLUMN IDENTIFICATION
# =====================================================================

def normalise_column_name(column_name):
    """
    Remove spaces and punctuation from a column name.
    """

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(column_name).lower()
    )


def find_column_case_insensitive(
    dataframe,
    possible_names
):
    """
    Find a dataframe column using several possible names.
    """

    available_columns = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }


    for possible_name in possible_names:

        normalised_name = normalise_column_name(
            possible_name
        )

        if normalised_name in available_columns:

            return available_columns[
                normalised_name
            ]


    return None

In [9]:
# =====================================================================
# 8. READ ALL FOUR CSV FILES
# =====================================================================

frames = []
raw_file_summary = []


for dataset_name, file_path in FILES.items():

    print(
        f"\nReading {dataset_name}: "
        f"{file_path.name}"
    )

    dataframe = pd.read_csv(
        file_path,
        low_memory=False
    )


    # Strip spaces from column names.
    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]


    # Find and standardise the Class column.
    class_column = find_column_case_insensitive(
        dataframe,
        ["Class"]
    )


    if class_column is None:

        raise ValueError(
            f"Class column was not found in "
            f"{file_path.name}"
        )


    if class_column != "Class":

        dataframe = dataframe.rename(
            columns={
                class_column: "Class"
            }
        )


    # Find and standardise the Label column.
    label_column = find_column_case_insensitive(
        dataframe,
        ["Label"]
    )


    if label_column is None:

        dataframe["Label"] = "Benign"

    elif label_column != "Label":

        dataframe = dataframe.rename(
            columns={
                label_column: "Label"
            }
        )


    dataframe["DatasetSource"] = dataset_name


    numeric_dataframe = dataframe.select_dtypes(
        include=[np.number]
    )


    if numeric_dataframe.shape[1] > 0:

        infinite_values = int(
            np.isinf(
                numeric_dataframe.to_numpy(
                    dtype=float
                )
            ).sum()
        )

    else:

        infinite_values = 0


    raw_file_summary.append({
        "Dataset": dataset_name,
        "Filename": file_path.name,
        "Rows": len(dataframe),
        "Columns": dataframe.shape[1],
        "MissingValues":
            int(dataframe.isna().sum().sum()),
        "InfiniteValues":
            infinite_values,
        "ExactFullRowDuplicates":
            int(dataframe.duplicated().sum())
    })


    frames.append(dataframe)


raw_summary_df = pd.DataFrame(
    raw_file_summary
)


raw_summary_df.to_csv(
    OUTPUT_DIR / "raw_file_summary.csv",
    index=False
)


print("\nRaw-file summary:")

display(
    raw_summary_df
)


raw_data = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)


print(
    "\nCombined raw-data shape:",
    raw_data.shape
)


Reading Benign: Benign.csv

Reading DNSCat2: Dnscat2.csv

Reading DNSTT: Dnstt.csv

Reading Iodine: Iodine.csv

Raw-file summary:


,Dataset,Filename,Rows,Columns,MissingValues,InfiniteValues,ExactFullRowDuplicates
0,Benign,Benign.csv,95234,37,0,0,0
1,DNSCat2,Dnscat2.csv,105009,37,0,0,0
2,DNSTT,Dnstt.csv,90334,37,0,0,0
3,Iodine,Iodine.csv,140000,37,0,0,0



Combined raw-data shape: (430577, 37)


In [10]:
# =====================================================================
# 9. NORMALISE CLASS AND TOOL LABELS
# =====================================================================

raw_data["Class"] = (
    raw_data["Class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "normal": "benign",
        "attack": "malicious"
    })
    .str.title()
)


def standardise_tool_label(value):
    """
    Standardise tool-name spellings.
    """

    cleaned_value = str(value).strip().lower()


    if "benign" in cleaned_value:
        return "Benign"


    if "dnscat" in cleaned_value:
        return "DNSCat2"


    if "dnstt" in cleaned_value:
        return "DNSTT"


    if "iodine" in cleaned_value:
        return "Iodine"


    return str(value).strip()


raw_data["Label"] = (
    raw_data["Label"]
    .apply(standardise_tool_label)
)


# Force every benign row to use the Benign tool label.
raw_data.loc[
    raw_data["Class"]
    .str.lower()
    .eq("benign"),
    "Label"
] = "Benign"


print("\nClass values:")

print(
    raw_data["Class"]
    .value_counts(dropna=False)
)


print("\nTool-label values:")

print(
    raw_data["Label"]
    .value_counts(dropna=False)
)


Class values:
Class
Malicious    335343
Benign        95234
Name: count, dtype: int64

Tool-label values:
Label
Iodine     140000
DNSCat2    105009
Benign      95234
DNSTT       90334
Name: count, dtype: int64


In [11]:
# =====================================================================
# 10. REMOVE SOURCE IP, DESTINATION IP, PORTS AND TIMESTAMP
# =====================================================================

columns_to_exclude = [
    "SourceIP",
    "Source IP",
    "SrcIP",
    "Src IP",

    "DestinationIP",
    "Destination IP",
    "DestIP",
    "DstIP",
    "Dst IP",

    "SourcePort",
    "Source Port",
    "SrcPort",
    "Src Port",

    "DestinationPort",
    "Destination Port",
    "DestPort",
    "DstPort",
    "Dst Port",

    "TimeStamp",
    "Timestamp",
    "Time Stamp",
    "DateTime",
    "Date Time",
    "CaptureTime",
    "Capture Time",

    "DatasetSource"
]


normalised_excluded_names = {
    normalise_column_name(column)
    for column in columns_to_exclude
}


actual_columns_to_remove = [
    column
    for column in raw_data.columns
    if normalise_column_name(column)
    in normalised_excluded_names
]


print("\nColumns removed:")

for column in actual_columns_to_remove:
    print(" -", column)


data = raw_data.drop(
    columns=actual_columns_to_remove,
    errors="ignore"
).copy()


Columns removed:
 - SourceIP
 - DestinationIP
 - SourcePort
 - DestinationPort
 - TimeStamp
 - DatasetSource


In [12]:
# =====================================================================
# 11. IDENTIFY FEATURE COLUMNS
# =====================================================================

TARGET_COLUMNS = [
    "Class",
    "Label"
]


FEATURE_COLUMNS = [
    column
    for column in data.columns
    if column not in TARGET_COLUMNS
]


print(
    "\nNumber of retained model features:",
    len(FEATURE_COLUMNS)
)


print("\nFeature columns:")

for feature_number, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):

    print(
        f"{feature_number:02d}. {feature}"
    )


Number of retained model features: 29

Feature columns:
01. Duration
02. FlowBytesSent
03. FlowSentRate
04. FlowBytesReceived
05. FlowReceivedRate
06. PacketLengthVariance
07. PacketLengthStandardDeviation
08. PacketLengthMean
09. PacketLengthMedian
10. PacketLengthMode
11. PacketLengthSkewFromMedian
12. PacketLengthSkewFromMode
13. PacketLengthCoefficientofVariation
14. PacketTimeVariance
15. PacketTimeStandardDeviation
16. PacketTimeMean
17. PacketTimeMedian
18. PacketTimeMode
19. PacketTimeSkewFromMedian
20. PacketTimeSkewFromMode
21. PacketTimeCoefficientofVariation
22. ResponseTimeTimeVariance
23. ResponseTimeTimeStandardDeviation
24. ResponseTimeTimeMean
25. ResponseTimeTimeMedian
26. ResponseTimeTimeMode
27. ResponseTimeTimeSkewFromMedian
28. ResponseTimeTimeSkewFromMode
29. ResponseTimeTimeCoefficientofVariation


In [13]:
# =====================================================================
# 12. CONVERT FEATURES TO NUMERIC
# =====================================================================

for feature in FEATURE_COLUMNS:

    data[feature] = pd.to_numeric(
        data[feature],
        errors="coerce"
    )


# Convert infinity to missing values.
data[FEATURE_COLUMNS] = (
    data[FEATURE_COLUMNS]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

In [14]:
# =====================================================================
# 13. HANDLE MISSING VALUES
# =====================================================================

missing_values_before = int(
    data[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)


print(
    "\nMissing values before imputation:",
    missing_values_before
)


for feature in FEATURE_COLUMNS:

    if data[feature].isna().any():

        feature_mean = data[feature].mean()


        if pd.isna(feature_mean):
            feature_mean = 0.0


        data[feature] = (
            data[feature]
            .fillna(feature_mean)
        )


missing_values_after = int(
    data[FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)


print(
    "Missing values after imputation :",
    missing_values_after
)


Missing values before imputation: 0
Missing values after imputation : 0


In [15]:
# =====================================================================
# 14. REMOVE DUPLICATES AFTER IDENTIFIERS ARE REMOVED
# =====================================================================

rows_before_duplicate_removal = len(
    data
)


data = (
    data
    .drop_duplicates()
    .reset_index(drop=True)
)


rows_after_duplicate_removal = len(
    data
)


duplicates_removed = (
    rows_before_duplicate_removal
    - rows_after_duplicate_removal
)


print(
    "\nRows before duplicate removal:",
    rows_before_duplicate_removal
)

print(
    "Rows after duplicate removal :",
    rows_after_duplicate_removal
)

print(
    "Duplicate rows removed       :",
    duplicates_removed
)


Rows before duplicate removal: 430577
Rows after duplicate removal : 407623
Duplicate rows removed       : 22954


In [16]:
# =====================================================================
# 15. REMOVE CONTRADICTORY FEATURE VECTORS
# =====================================================================
#
# A contradictory feature vector has exactly the same model features
# but different class or tool labels.
# Such rows can create unavoidable label conflict.
# =====================================================================

data["FeatureVectorHash"] = (
    pd.util.hash_pandas_object(
        data[FEATURE_COLUMNS],
        index=False
    )
    .astype(str)
)


data["CombinedTarget"] = (
    data["Class"].astype(str)
    + "_"
    + data["Label"].astype(str)
)


target_count_per_hash = (
    data
    .groupby("FeatureVectorHash")
    ["CombinedTarget"]
    .nunique()
)


contradictory_hashes = (
    target_count_per_hash[
        target_count_per_hash > 1
    ]
    .index
)


contradictory_rows = int(
    data["FeatureVectorHash"]
    .isin(contradictory_hashes)
    .sum()
)


print(
    "\nContradictory feature groups:",
    len(contradictory_hashes)
)

print(
    "Rows in contradictory groups:",
    contradictory_rows
)


if contradictory_rows > 0:

    data = data.loc[
        ~data["FeatureVectorHash"]
        .isin(contradictory_hashes)
    ].copy()


data = (
    data
    .drop(
        columns=["CombinedTarget"]
    )
    .reset_index(drop=True)
)


print(
    "Rows retained after contradiction removal:",
    len(data)
)


Contradictory feature groups: 1
Rows in contradictory groups: 2
Rows retained after contradiction removal: 407621


In [17]:
# =====================================================================
# 16. POST-PREPROCESSING SUMMARY
# =====================================================================

post_preprocessing_summary = pd.DataFrame([{
    "RawRows":
        len(raw_data),

    "RowsBeforeDuplicateRemoval":
        rows_before_duplicate_removal,

    "RowsAfterDuplicateRemoval":
        rows_after_duplicate_removal,

    "DuplicatesRemoved":
        duplicates_removed,

    "ContradictoryRowsRemoved":
        contradictory_rows,

    "FinalRows":
        len(data),

    "RetainedFeatures":
        len(FEATURE_COLUMNS),

    "MissingValues":
        int(
            data[FEATURE_COLUMNS]
            .isna()
            .sum()
            .sum()
        ),

    "InfiniteValues":
        int(
            np.isinf(
                data[FEATURE_COLUMNS]
                .to_numpy(dtype=float)
            ).sum()
        )
}])


post_preprocessing_summary.to_csv(
    OUTPUT_DIR /
    "post_preprocessing_summary.csv",
    index=False
)


print("\nPost-preprocessing summary:")

display(
    post_preprocessing_summary
)


Post-preprocessing summary:


,RawRows,RowsBeforeDuplicateRemoval,RowsAfterDuplicateRemoval,DuplicatesRemoved,ContradictoryRowsRemoved,FinalRows,RetainedFeatures,MissingValues,InfiniteValues
0,430577,430577,407623,22954,2,407621,29,0,0


In [18]:
# =====================================================================
# 17. SAVE FEATURE STATISTICS
# =====================================================================

feature_summary = pd.DataFrame(
    index=FEATURE_COLUMNS
)


feature_summary["Minimum"] = (
    data[FEATURE_COLUMNS]
    .min()
)


feature_summary["Maximum"] = (
    data[FEATURE_COLUMNS]
    .max()
)


feature_summary["Mean"] = (
    data[FEATURE_COLUMNS]
    .mean()
)


feature_summary["Median"] = (
    data[FEATURE_COLUMNS]
    .median()
)


feature_summary["StandardDeviation"] = (
    data[FEATURE_COLUMNS]
    .std()
)


feature_summary["Skewness"] = (
    data[FEATURE_COLUMNS]
    .skew()
)


feature_summary["UniqueValues"] = (
    data[FEATURE_COLUMNS]
    .nunique()
)


feature_summary["Minus1Count"] = (
    data[FEATURE_COLUMNS] == -1
).sum()


feature_summary["Minus10Count"] = (
    data[FEATURE_COLUMNS] == -10
).sum()


feature_summary.index.name = "Feature"


feature_summary.reset_index().to_csv(
    OUTPUT_DIR /
    "post_preprocessing_feature_summary.csv",
    index=False
)

In [19]:
# =====================================================================
# 18. GENERATE KDE-ONLY FEATURE PLOTS
# =====================================================================

random_generator = np.random.default_rng(
    RANDOM_STATE
)


print("\nGenerating KDE plots...")


for feature_number, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):

    all_values = (
        data[feature]
        .to_numpy(dtype=float)
    )


    all_values = all_values[
        np.isfinite(all_values)
    ]


    if len(all_values) < 2:

        print(
            f"Skipping {feature}: "
            "insufficient values"
        )

        continue


    # Display limits are computed PER CLASS and then combined:
    # lower limit = minimum of each class's 0.5th percentile,
    # upper limit = maximum of each class's 99.5th percentile.
    # This guarantees no class's central mass is cut at the plot
    # edge, even when class sizes are imbalanced, while single
    # extreme outliers still cannot stretch the axis.
    # This clipping is for plots only.
    per_class_lower_limits = []
    per_class_upper_limits = []


    for label in LABEL_ORDER:

        class_values = (
            data.loc[
                data["Label"].eq(label),
                feature
            ]
            .to_numpy(dtype=float)
        )


        class_values = class_values[
            np.isfinite(class_values)
        ]


        if len(class_values) < 2:
            continue


        per_class_lower_limits.append(
            np.quantile(
                class_values,
                0.005
            )
        )


        per_class_upper_limits.append(
            np.quantile(
                class_values,
                0.995
            )
        )


    if len(per_class_lower_limits) > 0:

        lower_limit = min(
            per_class_lower_limits
        )

        upper_limit = max(
            per_class_upper_limits
        )

    else:

        lower_limit = np.quantile(
            all_values,
            0.005
        )

        upper_limit = np.quantile(
            all_values,
            0.995
        )


    if (
        not np.isfinite(lower_limit)
        or not np.isfinite(upper_limit)
    ):

        lower_limit = np.min(
            all_values
        )

        upper_limit = np.max(
            all_values
        )


    if lower_limit == upper_limit:

        lower_limit -= 0.5
        upper_limit += 0.5


    x_grid = np.linspace(
        lower_limit,
        upper_limit,
        500
    )


    figure, axis = plt.subplots(
        figsize=(10.0, 7.0)
    )


    plotted_curve_count = 0


    for label in LABEL_ORDER:

        label_values = (
            data.loc[
                data["Label"].eq(label),
                feature
            ]
            .to_numpy(dtype=float)
        )


        label_values = label_values[
            np.isfinite(label_values)
        ]


        # Retain values within the visualisation interval.
        label_values = label_values[
            (label_values >= lower_limit)
            &
            (label_values <= upper_limit)
        ]


        if len(label_values) < 2:
            continue


        # Use a random sample for KDE estimation.
        if len(label_values) > MAX_KDE_SAMPLES:

            selected_indices = (
                random_generator.choice(
                    len(label_values),
                    size=MAX_KDE_SAMPLES,
                    replace=False
                )
            )


            kde_values = label_values[
                selected_indices
            ]

        else:

            kde_values = label_values


        # Gaussian KDE cannot process a fully constant array.
        if np.std(kde_values) == 0:

            jitter_standard_deviation = max(
                abs(kde_values[0]) * 1e-8,
                1e-8
            )


            kde_values = (
                kde_values
                +
                random_generator.normal(
                    loc=0.0,
                    scale=jitter_standard_deviation,
                    size=len(kde_values)
                )
            )


        try:

            kde_model = gaussian_kde(
                kde_values,
                bw_method="scott"
            )


            density = kde_model(
                x_grid
            )


            axis.plot(
                x_grid,
                density,
                color=KDE_COLOURS[label],
                linewidth=3.0,
                label=label
            )


            plotted_curve_count += 1


        except (
            np.linalg.LinAlgError,
            ValueError
        ) as error:

            print(
                f"Could not estimate KDE for "
                f"{feature}, {label}: {error}"
            )


    axis.set_xlabel(
        feature
    )


    axis.set_ylabel(
        "Density"
    )


    axis.set_xlim(
        lower_limit,
        upper_limit
    )


    axis.grid(False)


    axis.spines["top"].set_visible(
        False
    )


    axis.spines["right"].set_visible(
        False
    )


    # Place legend below the graph.
    if plotted_curve_count > 0:

        axis.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.20),
            ncol=4,
            frameon=False,
            columnspacing=1.8,
            handlelength=2.5
        )


    # No title is added.
    figure.subplots_adjust(
        bottom=0.27
    )


    safe_feature_name = re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        feature
    )


    kde_output_path = (
        KDE_DIR
        /
        f"{feature_number:02d}_"
        f"{safe_feature_name}_kde.pdf"
    )


    figure.savefig(
        kde_output_path,
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )


    plt.close(
        figure
    )


    print(
        f"{feature_number:02d}/"
        f"{len(FEATURE_COLUMNS)} "
        f"saved: {kde_output_path.name}"
    )


print(
    "\nNumber of KDE PDFs:",
    len(list(KDE_DIR.glob("*.pdf")))
)


Generating KDE plots...
01/29 saved: 01_Duration_kde.pdf
02/29 saved: 02_FlowBytesSent_kde.pdf
03/29 saved: 03_FlowSentRate_kde.pdf
04/29 saved: 04_FlowBytesReceived_kde.pdf
05/29 saved: 05_FlowReceivedRate_kde.pdf
06/29 saved: 06_PacketLengthVariance_kde.pdf
07/29 saved: 07_PacketLengthStandardDeviation_kde.pdf
08/29 saved: 08_PacketLengthMean_kde.pdf
09/29 saved: 09_PacketLengthMedian_kde.pdf
10/29 saved: 10_PacketLengthMode_kde.pdf
11/29 saved: 11_PacketLengthSkewFromMedian_kde.pdf
12/29 saved: 12_PacketLengthSkewFromMode_kde.pdf
13/29 saved: 13_PacketLengthCoefficientofVariation_kde.pdf
14/29 saved: 14_PacketTimeVariance_kde.pdf
15/29 saved: 15_PacketTimeStandardDeviation_kde.pdf
16/29 saved: 16_PacketTimeMean_kde.pdf
17/29 saved: 17_PacketTimeMedian_kde.pdf
18/29 saved: 18_PacketTimeMode_kde.pdf
19/29 saved: 19_PacketTimeSkewFromMedian_kde.pdf
20/29 saved: 20_PacketTimeSkewFromMode_kde.pdf
21/29 saved: 21_PacketTimeCoefficientofVariation_kde.pdf
22/29 saved: 22_ResponseTimeTimeVa

In [20]:
# =====================================================================
# 19. SAVE THE CLEANED COMBINED DATASET (HAND-OFF TO classification.ipynb)
# =====================================================================
#
# The cleaned, deduplicated, contradiction-free dataset is saved once
# here so classification.ipynb can load it directly and both notebooks
# are guaranteed to operate on identical rows.
# =====================================================================

data.to_csv(
    CLEANED_DATA_DIR /
    "combined_cleaned_data.csv.gz",
    index=False,
    compression="gzip"
)

print(
    "Saved cleaned dataset:",
    CLEANED_DATA_DIR / "combined_cleaned_data.csv.gz"
)

print(
    "Rows:",
    len(data),
    "| Features:",
    len(FEATURE_COLUMNS)
)


Saved cleaned dataset: /content/drive/MyDrive/Objective2/KDE/cleaned_data/combined_cleaned_data.csv.gz
Rows: 407621 | Features: 29


In [21]:
# =====================================================================
# 20. CREATE KDE ZIP FILE
# =====================================================================

kde_zip_path = shutil.make_archive(
    base_name=str(
        BASE_DIR /
        "kde_plots_600dpi"
    ),
    format="zip",
    root_dir=str(KDE_DIR)
)

print("\nKDE ZIP file:")
print(kde_zip_path)



KDE ZIP file:
/content/drive/MyDrive/Objective2/kde_plots_600dpi.zip
